# Lab 4 · Lakeflow Jobs: tasks, Run if, retries and repair

Build a three-task job from this notebook, make the first task fail on purpose, see Upstream failed and an All done cleanup task, repair the run with a new parameter, then add a weekday Quartz schedule (paused).

**Exam:** Databricks Certified Data Engineer Associate · **Domain:** Working with Lakeflow Jobs (16% of the exam) · **Time:** about 30 minutes

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end. Everything is created in the `workspace.dea_lab` schema, away from your other work.

From the Learning Hub certification guide: [Data Engineer Associate](https://www.datapreplab.com/hub#certifications/data-engineer-associate) · Independent study resource, not affiliated with or endorsed by Databricks.

## How this lab works

This notebook uses the **Databricks SDK for Python**, which is already installed in Databricks notebooks, to create three tiny task notebooks and one job. You then watch every run in the **Jobs & Pipelines** UI. Everything you do here can also be done by clicking in the UI; the code just makes the steps exact.

```text
ingest ──► publish ──► cleanup
           (All succeeded)   (Run if: All done)
```

## Part 1 · Three task notebooks

In [ ]:
import base64, time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs, workspace

w = WorkspaceClient()                       # uses this notebook's own sign-in
me = w.current_user.me().user_name
folder = f"/Users/{me}/dea_lab_jobs"
w.workspace.mkdirs(folder)

TASKS = {
    "ingest": '''# Databricks notebook source
dbutils.widgets.text("fail", "false")
if dbutils.widgets.get("fail") == "true":
    raise Exception("Simulated failure: the source system is not ready")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.dea_lab")
spark.sql("CREATE OR REPLACE TABLE workspace.dea_lab.job_orders AS SELECT id AS order_id, id * 10 AS amount FROM range(1, 6)")
print("ingest: 5 orders written")''',
    "publish": '''# Databricks notebook source
total = spark.sql("SELECT sum(amount) AS revenue FROM workspace.dea_lab.job_orders").first()["revenue"]
spark.sql(f"CREATE OR REPLACE TABLE workspace.dea_lab.job_revenue AS SELECT {total} AS revenue")
print("publish: revenue", total)''',
    "cleanup": '''# Databricks notebook source
print("cleanup: always runs, whatever happened upstream")''',
}
for name, source in TASKS.items():
    w.workspace.import_(path=f"{folder}/{name}", format=workspace.ImportFormat.SOURCE,
                        language=workspace.Language.PYTHON, overwrite=True,
                        content=base64.b64encode(source.encode()).decode())
print("Task notebooks are in", folder)

## Part 2 · Create the job

`fail` is a **job parameter**. Jobs pass job parameters to notebook tasks, where `dbutils.widgets.get("fail")` reads them. No compute is specified, so the tasks run on **serverless jobs compute**.

In [ ]:
def notebook(name):
    return jobs.NotebookTask(notebook_path=f"{folder}/{name}")

for old in w.jobs.list(name="dea_lab_daily_orders"):     # rerunning the lab replaces the job
    w.jobs.delete(old.job_id)

job = w.jobs.create(
    name="dea_lab_daily_orders",
    parameters=[jobs.JobParameterDefinition(name="fail", default="true")],
    tasks=[
        jobs.Task(task_key="ingest", notebook_task=notebook("ingest"),
                  max_retries=1, min_retry_interval_millis=10_000),
        jobs.Task(task_key="publish", notebook_task=notebook("publish"),
                  depends_on=[jobs.TaskDependency(task_key="ingest")]),
        jobs.Task(task_key="cleanup", notebook_task=notebook("cleanup"),
                  depends_on=[jobs.TaskDependency(task_key="publish")],
                  run_if=jobs.RunIf.ALL_DONE),
    ])
job_id = job.job_id
print(f"Job created: open Jobs & Pipelines -> dea_lab_daily_orders (job id {job_id})")

## Part 3 · Run it and watch it fail

**Predict:** what state will each task end in? `fail` defaults to `true`, and ingest has one retry.

In [ ]:
DONE = (jobs.RunLifeCycleState.TERMINATED, jobs.RunLifeCycleState.SKIPPED, jobs.RunLifeCycleState.INTERNAL_ERROR)

def wait_for(run_id, restarted=False):
    """Poll the run until it finishes, then print each task attempt's result."""
    if restarted:                               # a repaired run reports its old, finished state for a moment
        for _ in range(30):
            if w.jobs.get_run(run_id).state.life_cycle_state not in DONE:
                break
            time.sleep(2)
    while (run := w.jobs.get_run(run_id)).state.life_cycle_state not in DONE:
        time.sleep(10)
    print("Run result:", run.state.result_state.value if run.state.result_state else run.state.life_cycle_state.value)
    for task in sorted(run.tasks, key=lambda t: (t.start_time or 0, t.attempt_number or 0)):
        state = task.state.result_state or task.state.life_cycle_state
        print(f"  {task.task_key:<8} {state.value:<16} attempt {task.attempt_number}")
    return run

run_id = w.jobs.run_now(job_id=job_id).response.run_id
print("Started run", run_id, "- open it in the Jobs UI to watch the graph")
first = wait_for(run_id)

Open the run in the UI. Ingest failed twice (the first try and one retry), publish shows **Upstream failed** and never ran, and cleanup ran because its Run if is **All done**.

## Part 4 · Repair the run

A **repair** reruns only the tasks you choose, inside the same run, instead of starting from scratch. Here we fix the cause (set `fail` to `false`) and rerun ingest plus everything that depends on it.

In [ ]:
w.jobs.repair_run(run_id=run_id, rerun_tasks=["ingest"], rerun_dependent_tasks=True,
                  job_parameters={"fail": "false"})
repaired = wait_for(run_id, restarted=True)
display(spark.table("workspace.dea_lab.job_revenue"))

In the UI, the run now shows a **repair** in its history, and the matrix view shows each attempt.

## Part 5 · A weekday schedule, paused

Quartz cron has a **seconds** field first. `0 30 6 ? * MON-FRI` means 06:30:00 every weekday.

In [ ]:
w.jobs.update(job_id, new_settings=jobs.JobSettings(schedule=jobs.CronSchedule(
    quartz_cron_expression="0 30 6 ? * MON-FRI", timezone_id="Europe/London",
    pause_status=jobs.PauseStatus.PAUSED)))
print(w.jobs.get(job_id).settings.schedule)

Open the job's **Schedules & Triggers** panel to see the same schedule. The trigger types there are **Scheduled**, **File arrival**, **Table update** and **Continuous**.

## Part 6 · Explore in the UI (no code)

1. On the job page, click **+ Add task** after `publish` and choose **If/else condition**: compare `{{job.parameters.fail}}` with `false`. Notice the two outgoing branches, **true** and **false**.
2. Add a **For each** task and type `["UK","IE","FR"]` as its inputs; its nested task runs once per value.
3. Open **Run if dependencies** on `cleanup` and read the other options: At least one succeeded, None failed, All failed, At least one failed.

## Part 7 · Clean up

In [ ]:
w.jobs.delete(job_id)
w.workspace.delete(folder, recursive=True)
print("Job and task notebooks deleted")

## Expected results

```text
Part 3: Run result FAILED. ingest FAILED (you may see it twice: attempt 0 and the retry, attempt 1),
        publish UPSTREAM_FAILED, cleanup SUCCESS.
Part 4: after the repair, ingest, publish and cleanup are SUCCESS and the run result is SUCCESS.
        job_revenue shows 150 (10 + 20 + 30 + 40 + 50).
Part 5: the printed schedule has quartz_cron_expression '0 30 6 ? * MON-FRI' and pause_status PAUSED.
```

## Exam links

- A task that must always run, such as cleanup or a notification, needs Run if **All done**.
- Repair reruns only failed or chosen tasks and their dependants; it does not repeat tasks that succeeded.
- Unpredictable file arrivals: a file arrival trigger, not a frequent schedule that polls.
- Quartz cron: seconds first. A five-field Unix cron string is the wrong answer.